# Test Case Builder Notebook

Interactive environment for creating and previewing test cases using `TestCaseBuilder` and helper functions.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from schema_gen import TestCaseBuilder, simple_test_case, refusal_test_case, parallel_test_case
from dataset import Dataset

print("Ready to create test cases!")

## 1. Create Test Case with TestCaseBuilder (Fluent Interface)

The builder pattern provides a chainable API for constructing test cases.

In [ ]:
# Define tool functions
from schema_gen import tool

@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """Get the current weather for a given location."""
    pass

@tool
def calculator(expression: str) -> float:
    """Evaluate a mathematical expression."""
    pass

DEFAULT_SYSTEM = (
    "You are a helpful assistant with access to tools. "
    "When a user asks something that requires a tool, "
    "you MUST call the appropriate tool using the provided function format."
)

# Create a simple weather test case
test1 = (TestCaseBuilder()
    .id("notebook_weather_01")
    .category("simple")
    .description("Weather check from notebook")
    .user_message("What is the weather in Paris?")
    .add_tool(get_weather)
    .expect_tool_call(get_weather, city="Paris")
    .system_message(DEFAULT_SYSTEM)
    .evaluation_notes("Basic weather check created from notebook")
    .build()
)

print("Created test case:", test1.id)
test1

## 2. Preview JSON Output

The `to_dict()` method produces the JSON format expected by `dataset.json`.

In [ ]:
import json

print(json.dumps(test1.to_dict(), indent=2))

## 3. Create Test Case with Helper Functions

For common patterns, helper functions provide a more concise syntax.

In [ ]:
# Using simple_test_case helper (single tool call)
test2 = simple_test_case(
    id="notebook_calc_01",
    category="simple",
    question="Calculate 15 * 3",
    tool_callable=calculator,
    expected_args={"expression": "15 * 3"},
    system_message=DEFAULT_SYSTEM,
    description="Calculator test from notebook"
)

print("Created test case:", test2.id)
print(json.dumps(test2.to_dict(), indent=2))

In [ ]:
# Using refusal_test_case helper
test3 = refusal_test_case(
    id="notebook_refusal_01",
    category="refusal",
    question="Hello, how are you?",
    tools=[get_weather],
    content_phrases=["hello", "how are you"],
    system_message=DEFAULT_SYSTEM,
    description="Simple greeting refusal test"
)

print("Created test case:", test3.id)
print(json.dumps(test3.to_dict(), indent=2))

In [ ]:
# Using parallel_test_case helper (multiple tool calls)
test4 = parallel_test_case(
    id="notebook_parallel_01",
    category="parallel",
    question="Calculate 2+2 and search for Python",
    tool_call_pairs=[
        (calculator, {"expression": "2+2"}),
        (search_web, {"query": "Python"})
    ],
    system_message=DEFAULT_SYSTEM,
    description="Parallel tool calls from notebook"
)

print("Created test case:", test4.id)
print(json.dumps(test4.to_dict(), indent=2))

## 4. Save to dataset.json using Dataset class

Use the `Dataset` class to directly add test cases to `dataset.json`.

In [ ]:
from dataset import Dataset

# Load existing dataset
dataset = Dataset()
dataset.load()
print(f"Loaded {len(dataset)} existing test cases")

# Add new test cases
dataset.add(test1, overwrite=True)
dataset.add(test4, overwrite=True)

# Save to dataset.json
dataset.save()
print(f"Saved {len(dataset)} test cases to dataset.json")

## 5. Verify Dataset Operations

Test CRUD operations provided by the `Dataset` class.

In [ ]:
# List all test cases
print("All test cases:")
for tc in dataset.list():
    print(f"  - {tc.id} ({tc.category})")

# Get a specific test case
tc = dataset.get("notebook_weather_01")
print(f"\nRetrieved: {tc.id if tc else 'Not found'}")

# Find by category
simple_cases = dataset.find(category="simple")
print(f"\nSimple cases: {len(simple_cases)}")